# Trader Behavior Insights: Fear/Greed × Hyperliquid

This notebook:
1) Loads both datasets
2) Cleans & standardizes fields
3) Merges trades with daily sentiment
4) Produces core EDA + statistical tests
5) Trains a simple model to explain win/loss using sentiment and controls
6) Exports figures/tables




In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from pathlib import Path

from src.utils import parse_dates_safe, standardize_cols, label_direction, safe_num

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

BASE = Path('.')
DATA = BASE / 'data'
OUT = BASE / 'outputs'
OUT.mkdir(exist_ok=True, parents=True)


## 1) Load Data

In [ ]:
sent_path = DATA / 'market_sentiment.csv'
trades_path = DATA / 'hyperliquid_trades.csv'

assert sent_path.exists(), f"Missing {sent_path}. Place the Fear/Greed csv there."
assert trades_path.exists(), f"Missing {trades_path}. Place the trades csv there."

sent = pd.read_csv(sent_path)
trades = pd.read_csv(trades_path)

sent = standardize_cols(sent)
trades = standardize_cols(trades)

## 2) Clean & Harmonize

In [ ]:

sent = parse_dates_safe(sent, ['date'])
trades = parse_dates_safe(trades, ['time'])

sent['date_only'] = pd.to_datetime(sent['date'].dt.date)
trades['date_only'] = pd.to_datetime(trades['time'].dt.date)

if 'classification' in sent.columns:
    sent['classification'] = sent['classification'].astype(str).str.strip().str.lower()
    sent['classification'] = sent['classification'].replace({'fear':'fear','greed':'greed'})
else:
    raise ValueError("Sentiment CSV must include a 'Classification' column.")


for c in ['execution_price','size','closedpnl','leverage','start_position']:
    if c in trades.columns:
        trades[c] = trades[c].apply(safe_num)
        

if 'side' in trades.columns:
    trades['dir'] = trades.apply(label_direction, axis=1)
else:
    trades['dir'] = 0


trades = trades.dropna(subset=['execution_price', 'size'], how='any')
trades = trades[trades['size'] != 0]


## 3) Merge on Date

In [ ]:
merged = trades.merge(sent[['date_only','classification']], on='date_only', how='left')
merged['classification'] = merged['classification'].fillna('unknown')

if 'closedpnl' in merged.columns and merged['closedpnl'].notna().any():
    merged['pnl'] = merged['closedpnl']
else:
   
    merged['pnl'] = 0.0

merged['win'] = (merged['pnl'] > 0).astype(int)


merged.to_parquet(OUT / 'merged_trades_sentiment.parquet', index=False)
merged.head()


## 4) Core EDA

In [ ]:
def group_stats(df, by='classification'):
    g = df.groupby(by).agg(
        trades=('pnl','size'),
        win_rate=('win','mean'),
        avg_pnl=('pnl','mean'),
        p95_loss=('pnl', lambda x: np.percentile(x, 5)),
        p95_gain=('pnl', lambda x: np.percentile(x, 95)),
        avg_leverage=('leverage','mean') if 'leverage' in df.columns else ('pnl','mean')
    )
    return g

summary_all = group_stats(merged)
summary_all.to_csv(OUT / 'summary_overall_by_sentiment.csv')
summary_all


In [ ]:

acc_sent = merged.groupby(['account','classification']).agg(
    trades=('pnl','size'),
    win_rate=('win','mean'),
    avg_pnl=('pnl','mean'),
    avg_lev=('leverage','mean') if 'leverage' in merged.columns else ('pnl','mean')
).reset_index()

acc_sent.to_csv(OUT / 'summary_per_account_by_sentiment.csv', index=False)
acc_sent.head()


### Visuals

In [ ]:

fig = plt.figure()
plot_df = merged.groupby('classification')['win'].mean().reset_index()
plt.bar(plot_df['classification'], plot_df['win'])
plt.title('Win Rate by Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Win Rate')
plt.savefig(OUT / 'win_rate_by_sentiment.png', bbox_inches='tight')
plt.show()


In [ ]:

fig = plt.figure()
fg = merged[merged['classification'].isin(['fear','greed'])]
data = [fg.loc[fg['classification']=='fear','pnl'].dropna(),
        fg.loc[fg['classification']=='greed','pnl'].dropna()]
plt.boxplot(data, labels=['fear','greed'])
plt.title('PnL Distribution by Sentiment')
plt.ylabel('PnL')
plt.savefig(OUT / 'pnl_box_by_sentiment.png', bbox_inches='tight')
plt.show()


## 5) Statistical Test (Greed vs Fear)

In [ ]:
fear = merged.loc[merged['classification']=='fear','pnl'].dropna()
greed = merged.loc[merged['classification']=='greed','pnl'].dropna()

if len(fear) > 2 and len(greed) > 2:
    
    t, p = stats.ttest_ind(greed, fear, equal_var=False, nan_policy='omit')
    print('Welch t-test Greed vs Fear PnL: t=%.3f p=%.3g' % (t, p))
else:
    print('Not enough data points in fear/greed buckets to run t-test.')


## 6) Simple Model: Win ~ Sentiment + Leverage + Symbol (dummies)

In [ ]:
model_df = merged.copy()
model_df = model_df[model_df['win'].isin([0,1])]


model_df['is_greed'] = (model_df['classification']=='greed').astype(int)
if 'leverage' not in model_df.columns:
    model_df['leverage'] = 0.0

top_syms = model_df['symbol'].value_counts().head(10).index if 'symbol' in model_df.columns else []
if 'symbol' in model_df.columns:
    model_df['symbol_f'] = model_df['symbol'].where(model_df['symbol'].isin(top_syms), 'OTHER')
    X = pd.get_dummies(model_df[['is_greed','leverage','symbol_f']], drop_first=True)
else:
    X = model_df[['is_greed','leverage']]

y = model_df['win']

if X.shape[0] > 100:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, y_train)
    y_prob = clf.predict_proba(X_test)[:,1]
    y_pred = (y_prob >= 0.5).astype(int)
    print(classification_report(y_test, y_pred, digits=3))
    print('ROC-AUC:', roc_auc_score(y_test, y_prob).round(3))

    coefs = pd.DataFrame({'feature': X.columns, 'coef': clf.coef_[0]}).sort_values('coef', ascending=False)
    coefs.to_csv(OUT / 'logit_coefs.csv', index=False)
    coefs.head(10)
else:
    print('Not enough rows for a stable train/test split; collect more data or loosen filters.')


## 7) Account Archetypes (optional)

In [ ]:

arch = merged.copy()
arch['lev_bucket'] = pd.qcut(arch['leverage'].fillna(0), q=4, duplicates='drop') if 'leverage' in arch.columns else 'NA'

ar_summary = arch.groupby(['account','lev_bucket']).agg(
    n=('pnl','size'),
    win_rate=('win','mean'),
    avg_pnl=('pnl','mean')
).reset_index()

ar_summary.to_csv(OUT / 'account_archetypes.csv', index=False)
ar_summary.head()


## 8) Save Key Tables

In [ ]:
summary_all.to_csv(OUT / 'summary_overall_by_sentiment.csv')
acc_sent.to_csv(OUT / 'summary_per_account_by_sentiment.csv', index=False)
print('Saved tables to outputs/.')